In [0]:
import pytest
from pyspark.sql import SparkSession, Row
from pyspark.sql import functions as F

def get_trip_event_rules():
    return [
        {"name": "trip_id_not_null", "condition": F.col("trip_id").isNotNull()},
        {"name": "non_negative_fare", "condition": F.col("fare_amount").isNull() | (F.col("fare_amount") >= 0)},
    ]


def test_negative_fares_flagged(spark):
    df = spark.createDataFrame([
        Row(trip_id="T1", fare_amount=-15.0),
        Row(trip_id="T2", fare_amount=20.0),
    ])
    rules = get_trip_event_rules()
    condition = rules[1]["condition"]
    result = df.withColumn("is_valid", condition).collect()
    assert result[0]["is_valid"] is False
    assert result[1]["is_valid"] is True


def test_missing_trip_id_flagged(spark):
    df = spark.createDataFrame([
        Row(trip_id=None, fare_amount=10.0),
        Row(trip_id="T2", fare_amount=10.0),
    ])
    rules = get_trip_event_rules()
    condition = rules[0]["condition"]
    result = df.withColumn("is_valid", condition).collect()
    assert result[0]["is_valid"] is False
    assert result[1]["is_valid"] is True

In [0]:
def run_test(test_func, name):
    try:
        test_func(spark)
        print(f"✅ PASS: {name}")
    except AssertionError as e:
        print(f"❌ FAIL: {name} — {e}")
    except Exception as e:
        print(f"⚠️ ERROR: {name} — {e}")

run_test(test_negative_fares_flagged, "test_negative_fares_flagged")
run_test(test_missing_trip_id_flagged, "test_missing_trip_id_flagged")